# Build lists of images and texts

In [1]:
import glob
from pathlib import Path

dataset_path = Path(r'C:\Users\Usuario\Downloads\PFC1\DTrOCR\iam_words')

xml_files = sorted(glob.glob(str(dataset_path / 'xml' / '*.xml')))
word_image_files = sorted(glob.glob(str(dataset_path / 'words' / '**' / '*.png'), recursive=True))

print(f"{len(xml_files)} XML files and {len(word_image_files)} word image files")

1539 XML files and 115320 word image files


In [ ]:
import tqdm
import multiprocessing as mp
import xml.etree.ElementTree as ET

from PIL import Image
from dataclasses import dataclass

from pathlib import Path

@dataclass
class Word:
    id: str
    file_path: Path
    writer_id: str
    transcription: str

def get_words_from_xml(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    root_id = root.get('id')
    writer_id = root.get('writer-id')
    xml_words = []
    for line in root.findall('handwritten-part')[0].findall('line'):
        for word in line.findall('word'):
            image_file = Path([f for f in word_image_files if f.endswith(word.get('id') + '.png')][0])
            try:
                with Image.open(image_file) as _:
                    xml_words.append(
                        Word(
                            id=root_id,
                            file_path=image_file,
                            writer_id=writer_id,
                            transcription=word.get('text')
                        )
                    )
            except Exception:
                pass
            
    return xml_words

# with mp.Pool(processes=mp.cpu_count()) as pool:
#     words_from_xmls = list(
#         tqdm.tqdm(
#             pool.imap(get_words_from_xml, xml_files), 
#             total=len(xml_files),
#             desc='Building dataset'
#         )
#     )
words_from_xmls = []
for xml_file in tqdm.tqdm(xml_files, desc='Building dataset'):
    words_from_xmls.append(get_words_from_xml(xml_file))

words = [word for words in words_from_xmls for word in words]

Building dataset:   0%|▎                                                              | 7/1539 [00:12<45:29,  1.78s/it]

In [ ]:
# EJECUTAR EL BLOQUE DE ABAJO SOLO SI SE GUARDO LAS PALABRAS.PKL 

In [2]:
import pickle
import tqdm
import multiprocessing as mp
import xml.etree.ElementTree as ET

from PIL import Image
from dataclasses import dataclass

from pathlib import Path

@dataclass
class Word:
    id: str
    file_path: Path
    writer_id: str
    transcription: str

# Cargamos la lista de objetos 'words.pkl'
with open('dataset_words.pkl', 'rb') as f:
    words = pickle.load(f)

print(f"{len(words)} palabras recuperadas")

115318 palabras recuperadas


# Train test split

In [3]:
with open(dataset_path / 'splits/train.uttlist') as fp:
    train_ids = [line.replace('\n', '') for line in fp.readlines()]

with open(dataset_path / 'splits/test.uttlist') as fp:
    test_ids = [line.replace('\n', '') for line in fp.readlines()]

with open(dataset_path / 'splits/validation.uttlist') as fp:
    validation_ids = [line.replace('\n', '') for line in fp.readlines()]

print(f"Train size: {len(train_ids)}; Validation size: {len(validation_ids)}; Test size: {len(test_ids)}")

Train size: 747; Validation size: 116; Test size: 336


In [4]:
train_word_records = [word for word in words if word.id in train_ids]
validation_word_records = [word for word in words if word.id in validation_ids]
test_word_records = [word for word in words if word.id in test_ids]

print(f'Train size: {len(train_word_records)}; Validation size: {len(validation_word_records)}; Test size: {len(test_word_records)}')

Train size: 55079; Validation size: 8895; Test size: 25920


# Build dataset and dataloader

In [5]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from dtrocr.processor import DTrOCRProcessor
from dtrocr.config import DTrOCRConfig

from torch.utils.data import Dataset

class IAMDataset(Dataset):
    def __init__(self, words: list[Word], config: DTrOCRConfig):
        super(IAMDataset, self).__init__()
        self.words = words
        self.processor = DTrOCRProcessor(config, add_eos_token=True, add_bos_token=True)
        
    def __len__(self):
        return len(self.words)
    
    def __getitem__(self, item):
        inputs = self.processor(
            images=Image.open(self.words[item].file_path).convert('RGB'),
            texts=self.words[item].transcription,
            padding='max_length',
            return_tensors="pt",
            return_labels=True,
            input_data_format='channels_first', # Fuerza los canales al principio (3, Alto, Ancho)
        )
        return {
            'pixel_values': inputs.pixel_values[0],
            'input_ids': inputs.input_ids[0],
            'attention_mask': inputs.attention_mask[0],
            'labels': inputs.labels[0]
        }

config = DTrOCRConfig(
    # attn_implementation='flash_attention_2'
)

train_data = IAMDataset(words=train_word_records, config=config)
validation_data = IAMDataset(words=validation_word_records, config=config)
test_data = IAMDataset(words=test_word_records, config=config)
print("okey")

okey


In [6]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_data, batch_size=8, shuffle=True, num_workers=0)
validation_dataloader = DataLoader(validation_data, batch_size=8, shuffle=False, num_workers=0)
test_dataloader = DataLoader(test_data, batch_size=8, shuffle=False, num_workers=0)
print("Okey")

Okey


# Model

In [8]:
import torch
torch.set_float32_matmul_precision('high')

from dtrocr.model import DTrOCRLMHeadModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Entrenando en: {device}")

# model = DTrOCRLMHeadModel(config)
# model = torch.compile(model)
# model.to(device=0)

model = DTrOCRLMHeadModel(config)
model.to(device)
print("okey")

Entrenando en: cuda
okey


In [13]:
import pickle

# Guarda la lista de objetos 'words' en un archivo local
with open('dataset_words.pkl', 'wb') as f:
    pickle.dump(words, f)

print("words guardado!")

words guardado!


# Training

In [13]:
import logging
import sys
import os

# Crear el logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("run.log", mode='a', encoding='utf-8'),
        logging.StreamHandler(sys.stdout)    ]
)
logging.info("Iniciando el proceso de entrenamiento DTrOCR...")

# training
from typing import Tuple

def evaluate_model(model: torch.nn.Module, dataloader: DataLoader) -> Tuple[float, float]:
    # set model to evaluation mode
    model.eval()
    
    losses, accuracies = [], []
    with torch.no_grad():
        for inputs in tqdm.tqdm(dataloader, total=len(dataloader), desc=f'Evaluating test set'):
            # inputs = send_inputs_to_device(inputs, device=0)
            inputs = send_inputs_to_device(inputs, device=device)
            outputs = model(**inputs)
            
            losses.append(outputs.loss.item())
            accuracies.append(outputs.accuracy.item())
    
    loss = sum(losses) / len(losses)
    accuracy = sum(accuracies) / len(accuracies)
    
    # set model back to training mode
    model.train()
    
    return loss, accuracy

def send_inputs_to_device(dictionary, device):
    return {key: value.to(device=device) if isinstance(value, torch.Tensor) else value for key, value in dictionary.items()}

use_amp = True
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
optimiser = torch.optim.Adam(params=model.parameters(), lr=1e-4)

EPOCHS = 50
train_losses, train_accuracies = [], []
validation_losses, validation_accuracies = [], []
for epoch in range(EPOCHS):
    epoch_losses, epoch_accuracies = [], []
    for inputs in tqdm.tqdm(train_dataloader, total=len(train_dataloader), desc=f'Epoch {epoch + 1}'):
        
        # set gradients to zero
        optimiser.zero_grad()
        
        # send inputs to same device as model
        # inputs = send_inputs_to_device(inputs, device=0)
        inputs = send_inputs_to_device(inputs, device=device)
        
        # forward pass
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
            outputs = model(**inputs)
        
        # calculate gradients
        scaler.scale(outputs.loss).backward()
        
        # update weights
        scaler.step(optimiser)
        scaler.update()
        
        epoch_losses.append(outputs.loss.item())
        epoch_accuracies.append(outputs.accuracy.item())
        
    # store loss and metrics
    train_losses.append(sum(epoch_losses) / len(epoch_losses))
    train_accuracies.append(sum(epoch_accuracies) / len(epoch_accuracies))
    
    # tests loss and accuracy
    validation_loss, validation_accuracy = evaluate_model(model, validation_dataloader)
    validation_losses.append(validation_loss)
    validation_accuracies.append(validation_accuracy)
                    
    # print(f"Epoch: {epoch + 1} - Train loss: {train_losses[-1]}, Train accuracy: {train_accuracies[-1]}, Validation loss: {validation_losses[-1]}, Validation accuracy: {validation_accuracies[-1]}")

    # # guardar pesos
    # import os
    # os.makedirs('output', exist_ok=True)
    # # Guardar el modelo al final de cada época
    # save_path = f"output/dtrocr_epoch_{epoch + 1}.pth"
    # torch.save(model.state_dict(), save_path)
    # print(f"Modelo guardado en: {save_path}")
    # Registrar métricas de la época en el log
    
    logging.info(
        f"Epoch: {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_losses[-1]:.4f} | Train Acc: {train_accuracies[-1]:.4f} | "
        f"Val Loss: {validation_losses[-1]:.4f} | Val Acc: {validation_accuracies[-1]:.4f}"
    )

    # Guardar pesos
    os.makedirs('output', exist_ok=True)
    save_path = f"output/dtrocr_epoch_{epoch + 1}.pth"
    torch.save(model.state_dict(), save_path)
    logging.info(f"Modelo guardado exitosamente en: {save_path}")
print("okey")

2026-05-27 02:16:18,888 - INFO - Iniciando el proceso de entrenamiento DTrOCR...


C:\Users\Usuario\AppData\Local\Temp\ipykernel_14156\2887170258.py:44: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
Evaluating test set: 100%|██████████████████████████████████████████████████████████| 1112/1112 [01:31<00:00, 12.14it/s]

2026-05-27 02:39:13,324 - INFO - Epoch: 1/50 | Train Loss: 3.3996 | Train Acc: 0.5413 | Val Loss: 2.9374 | Val Acc: 0.5968


2026-05-27 02:39:14,127 - INFO - Modelo guardado exitosamente en: output/dtrocr_epoch_1.pth


Evaluating test set: 100%|██████████████████████████████████████████████████████████| 1112/1112 [01:26<00:00, 12.78it/s]

...

Evaluating test set: 100%|██████████████████████████████████████████████████████████| 1112/1112 [01:34<00:00, 11.75it/s]


2026-05-27 20:02:13,671 - INFO - Epoch: 49/50 | Train Loss: 0.0195 | Train Acc: 0.9948 | Val Loss: 3.1607 | Val Acc: 0.6905
2026-05-27 20:02:14,542 - INFO - Modelo guardado exitosamente en: output/dtrocr_epoch_49.pth


Evaluating test set: 100%|██████████████████████████████████████████████████████████| 1112/1112 [01:26<00:00, 12.88it/s]

2026-05-27 20:25:36,296 - INFO - Epoch: 50/50 | Train Loss: 0.0192 | Train Acc: 0.9946 | Val Loss: 3.1174 | Val Acc: 0.6888


2026-05-27 20:25:37,109 - INFO - Modelo guardado exitosamente en: output/dtrocr_epoch_50.pth
okey


# Test

In [9]:
from dtrocr.model import DTrOCRLMHeadModel
from dtrocr.config import DTrOCRConfig
from dtrocr.processor import DTrOCRProcessor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model = DTrOCRLMHeadModel(DTrOCRConfig())
model.eval()
# model.to('cpu')
model.to(device)
# test_processor = DTrOCRProcessor(DTrOCRConfig())
test_processor = DTrOCRProcessor(DTrOCRConfig(), add_bos_token=True, add_eos_token=True)
print("okey")

okey


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
import tqdm
from jiwer import cer

ground_truths = []
predictions = []

print("Generando predicciones para el cálculo del CER...")
for test_word_record in tqdm.tqdm(test_word_records):
    image_file = test_word_record.file_path
    real_text = test_word_record.transcription
    
    # Prepara la imagen
    image = Image.open(image_file).convert('RGB')
    
    inputs = test_processor(
        images=image, 
        texts=test_processor.tokeniser.bos_token,
        return_tensors='pt',
        input_data_format='channels_first'
    )
    
    # inputs = {k: v.to(model.device) if hasattr(v, 'to') else v for k, v in inputs.items()}
    # inputs = {k: v.to(device) if hasattr(v, 'to') else v for k, v in inputs.items()}

    if inputs.pixel_values is not None:
        inputs.pixel_values = inputs.pixel_values.to(device)
    if inputs.input_ids is not None:
        inputs.input_ids = inputs.input_ids.to(device)
    if inputs.attention_mask is not None:
        inputs.attention_mask = inputs.attention_mask.to(device)
    
    # Generar texto
    model_output = model.generate(
        inputs, 
        test_processor, 
        num_beams=3, 
        use_cache=True
    )
    
    predicted_text = test_processor.tokeniser.decode(model_output[0], skip_special_tokens=True)
    
    ground_truths.append(real_text)
    predictions.append(predicted_text)

# Calcular el CER
error_rate = cer(ground_truths, predictions)

# print(f"\n======================================")
# print(f"Total de palabras evaluadas: {len(ground_truths)}")
# print(f"Character Error Rate (CER): {error_rate * 100:.2f}%")
# print(f"======================================")

logging.info("======================================")
logging.info(f"Evaluación Final Completada")
logging.info(f"Total de palabras evaluadas: {len(ground_truths)}")
logging.info(f"Character Error Rate (CER): {error_rate * 100:.2f}%")
logging.info("======================================")

print("\nMuestra de predicciones:")
for i in range(10):
    print(f"Real: '{ground_truths[i]}' | Predicción: '{predictions[i]}'")

# from PIL import Image

# import numpy as np
# import matplotlib.pyplot as plt

# from PIL import Image

# for test_word_record in test_word_records[:50]:
#     image_file = test_word_record.file_path
#     image = Image.open(image_file).convert('RGB')
    
#     inputs = test_processor(
#         images=image, 
#         texts=test_processor.tokeniser.bos_token,
#         return_tensors='pt'
#     )
    
#     model_output = model.generate(
#         inputs, 
#         test_processor, 
#         num_beams=3
#     )
    
#     predicted_text = test_processor.tokeniser.decode(model_output[0], skip_special_tokens=True)
    
#     plt.figure(figsize=(10, 5))
#     plt.title(predicted_text, fontsize=24)
#     plt.imshow(np.array(image, dtype=np.uint8))
#     plt.xticks([]), plt.yticks([])
#     plt.show()

In [ ]:
# TEST CON LOS PESOS CARGADOS EN OUTPUT

In [9]:
import torch
import os
from dtrocr.config import DTrOCRConfig
from dtrocr.model import DTrOCRLMHeadModel
from dtrocr.processor import DTrOCRProcessor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Evaluando en: {device}")

# inicializa la arquitectura base y el procesador
config = DTrOCRConfig()
test_processor = DTrOCRProcessor(config, add_bos_token=True, add_eos_token=True)
model = DTrOCRLMHeadModel(config)

# SELECCIONAR LA EPOCA A EVALUAR
epoca_a_evaluar = 7 

ruta_pesos = f"output/dtrocr_epoch_{epoca_a_evaluar}.pth"

# cargar los pesos  
model.load_state_dict(torch.load(ruta_pesos, map_location=device, weights_only=True))
model.to(device)

# modo inferencia apagar Dropout
model.eval()
print(f"Modelo de la época {epoca_a_evaluar} cargado con éxito.")

Evaluando en: cuda
Modelo de la época 7 cargado con éxito.


In [33]:
import numpy as np
import torch
import tqdm
from jiwer import cer
from PIL import Image

ground_truths = []
predictions = []

print(f"Generando predicciones para el CER (Epoca {epoca_a_evaluar})...")

with torch.no_grad():
    for test_word_record in tqdm.tqdm(test_word_records):
        image_file = test_word_record.file_path
        real_text = test_word_record.transcription
        
        image = Image.open(image_file).convert('RGB')
        
        inputs = test_processor(
            images=image, 
            return_tensors='pt',
            input_data_format='channels_first'                 
        )
        
        bos_id = test_processor.tokeniser.bos_token_id
        inputs.input_ids = torch.tensor([[bos_id]])
        inputs.attention_mask = torch.tensor([[1]])
        
        # enviar tensores a la GPU
        if inputs.pixel_values is not None:
            inputs.pixel_values = inputs.pixel_values.to(device)
        if inputs.input_ids is not None:
            inputs.input_ids = inputs.input_ids.to(device)
        if inputs.attention_mask is not None:
            inputs.attention_mask = inputs.attention_mask.to(device)
        
        # generar texto
        model_output = model.generate(
            inputs, 
            test_processor, 
            num_beams=3, 
            use_cache=True
        )
        
        predicted_text = test_processor.tokeniser.decode(model_output[0], skip_special_tokens=True)
        
        ground_truths.append(real_text)
        predictions.append(predicted_text)

# calcular el CER
error_rate = cer(ground_truths, predictions)

print(f"\n======================================")
print(f"RESULTADOS TESTING - EPOCA {epoca_a_evaluar}")
print(f"Total de palabras evaluadas: {len(ground_truths)}")
print(f"Character Error Rate (CER): {error_rate * 100:.2f}%")
print(f"======================================")

print("\nMuestra de predicciones:")
# 10 ejemplos para visualizar
for i in range(min(10, len(ground_truths))):
    print(f"Real: '{ground_truths[i]}' | Predicción: '{predictions[i]}'")

Generando predicciones para el CER (Epoca 50)...


100%|████████████████████████████████████████████████████████████████████████████| 25920/25920 [30:42<00:00, 14.07it/s]



RESULTADOS FINALES - EPOCA 50
Total de palabras evaluadas: 25920
Character Error Rate (CER): 59.09%

Muestra de predicciones:
Real: 'Become' | Predicción: 'Rome'
Real: 'a' | Predicción: 'a'
Real: 'success' | Predicción: 'was'
Real: 'with' | Predicción: 'with'
Real: 'a' | Predicción: 'a'
Real: 'disc' | Predicción: 'other'
Real: 'and' | Predicción: 'and'
Real: 'hey' | Predicción: 'they'
Real: 'presto' | Predicción: 'again'
Real: '!' | Predicción: '!'


In [44]:
print(f"\n======================================")
print(f"RESULTADOS TESTING - EPOCA {epoca_a_evaluar}")
print(f"Total de palabras evaluadas: {len(ground_truths)}")
print(f"Character Error Rate (CER): {error_rate * 100:.2f}%")
print(f"======================================")

print("\nMuestra de predicciones:")
# 10 ejemplos para visualizar
for i in range(min(10, len(ground_truths))):
    print(f"Real: '{ground_truths[i]}' | Predicción: '{predictions[i]}'")


RESULTADOS TESTING - EPOCA 50
Total de palabras evaluadas: 25920
Character Error Rate (CER): 0.00%

Muestra de predicciones:
Real: 'Become' | Predicción: 'Rome'
Real: 'a' | Predicción: 'a'
Real: 'success' | Predicción: 'was'
Real: 'with' | Predicción: 'with'
Real: 'a' | Predicción: 'a'
Real: 'disc' | Predicción: 'other'
Real: 'and' | Predicción: 'and'
Real: 'hey' | Predicción: 'they'
Real: 'presto' | Predicción: 'again'
Real: '!' | Predicción: '!'


In [43]:
import torch
from PIL import Image
from jiwer import cer
from dtrocr.config import DTrOCRConfig
from dtrocr.model import DTrOCRLMHeadModel
from dtrocr.processor import DTrOCRProcessor

epoca_a_evaluar = 50
# DTROCR
# ruta_imagen = r"C:\Users\Usuario\Downloads\PFC1\DTrOCR\iam_words\words\a01\a01-000u\a01-000u-01-05.png" 
# texto_real = "Peers"
ruta_imagen = r"C:/Users/Usuario/Downloads/PFC1/DTrOCR/iam_words/words/a01/a01-000u/a01-000u-00-01.png" 
texto_real = "MOVE"

# HTR-VT
# C:/Users/Usuario/Downloads/PFC1/HTR-VT/data/iam/lines/a01-000u-00.png     A MOVE to stop Mr. Gaitskell from
# C:/Users/Usuario/Downloads/PFC1/HTR-VT/data/iam/lines/p03-185-01.png     Diana was trailing up the gravelled drive to the hospital

# ruta_imagen = r"C:/Users/Usuario/Downloads/PFC1/HTR-VT/data/iam/lines/a01-000u-00.png " 
# texto_real = "A MOVE to stop Mr. Gaitskell from"




device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
config = DTrOCRConfig()
processor = DTrOCRProcessor(config, add_bos_token=True, add_eos_token=True)

model = DTrOCRLMHeadModel(config)
ruta_pesos = f"output/dtrocr_epoch_{epoca_a_evaluar}.pth"

model.load_state_dict(torch.load(ruta_pesos, map_location=device, weights_only=True))
model.to(device)
model.eval()

image = Image.open(ruta_imagen).convert('RGB')
inputs = processor(
    images=image, 
    return_tensors='pt',
    input_data_format='channels_first',
)

bos_id = processor.tokeniser.bos_token_id
inputs.input_ids = torch.tensor([[bos_id]])
inputs.attention_mask = torch.tensor([[1]])

if inputs.pixel_values is not None:
    inputs.pixel_values = inputs.pixel_values.to(device)
if inputs.input_ids is not None:
    inputs.input_ids = inputs.input_ids.to(device)
if inputs.attention_mask is not None:
    inputs.attention_mask = inputs.attention_mask.to(device)

print("Analizando imagen...")
with torch.no_grad():
    model_output = model.generate(
        inputs, 
        processor, 
        num_beams=3,  
        use_cache=True
    )

texto_predicho = processor.tokeniser.decode(model_output[0], skip_special_tokens=True)
error_rate = cer(texto_real, texto_predicho)

# Mostrar resultados
print("\n======================================")
print(f"Ruta de imagen : {ruta_imagen}")
print(f"Texto Real     : '{texto_real}'")
print(f"Texto Predicho : '{texto_predicho}'")
print(f"CER de la imagen: {error_rate * 100:.2f}%")
print("======================================")

Analizando imagen...

Ruta de imagen : C:/Users/Usuario/Downloads/PFC1/DTrOCR/iam_words/words/a01/a01-000u/a01-000u-00-01.png
Texto Real     : 'MOVE'
Texto Predicho : 'MOVE'
CER de la imagen: 0.00%


In [28]:
for i in range(10):
    print(train_word_records[i])

Word(id='a01-000u', file_path=WindowsPath('C:/Users/Usuario/Downloads/PFC1/DTrOCR/iam_words/words/a01/a01-000u/a01-000u-00-00.png'), writer_id='000', transcription='A')
Word(id='a01-000u', file_path=WindowsPath('C:/Users/Usuario/Downloads/PFC1/DTrOCR/iam_words/words/a01/a01-000u/a01-000u-00-01.png'), writer_id='000', transcription='MOVE')
Word(id='a01-000u', file_path=WindowsPath('C:/Users/Usuario/Downloads/PFC1/DTrOCR/iam_words/words/a01/a01-000u/a01-000u-00-02.png'), writer_id='000', transcription='to')
Word(id='a01-000u', file_path=WindowsPath('C:/Users/Usuario/Downloads/PFC1/DTrOCR/iam_words/words/a01/a01-000u/a01-000u-00-03.png'), writer_id='000', transcription='stop')
Word(id='a01-000u', file_path=WindowsPath('C:/Users/Usuario/Downloads/PFC1/DTrOCR/iam_words/words/a01/a01-000u/a01-000u-00-04.png'), writer_id='000', transcription='Mr.')
Word(id='a01-000u', file_path=WindowsPath('C:/Users/Usuario/Downloads/PFC1/DTrOCR/iam_words/words/a01/a01-000u/a01-000u-00-05.png'), writer_id='00